# 005_postprocess_within_spatial_condition_specific.ipynb

Within-condition spatial postprocessing for the condition-specific ranking branch. This uses the new decoding output directory prefix but otherwise preserves the within-condition logic.


In [ ]:
# ============================================================
# FIGURE SAVING SETTINGS -- EXPLICIT, NO RECURSION
# ============================================================

from pathlib import Path

SAVE_FIGS = True
FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "005_postprocess_within_spatial_refined"
FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_FORMAT = "pdf"
DPI = 300

FIG_DIR.mkdir(parents=True, exist_ok=True)

_fig_counter = 0

def _sanitize_fig_name(name):
    name = str(name).replace(" ", "_").replace("|", "_").replace("/", "-").replace("\\", "-")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:160] if name else "figure"

def _figure_has_content(fig=None):
    if fig is None:
        fig = plt.gcf()
    if len(fig.axes) == 0:
        return False
    for ax in fig.axes:
        if ax.lines or ax.collections or ax.images or ax.patches or ax.texts or ax.get_title():
            return True
    return True

def save_current_fig(name=None):
    global _fig_counter
    if not SAVE_FIGS:
        return None

    fig = plt.gcf()
    if not _figure_has_content(fig):
        return None

    _fig_counter += 1

    if name is None:
        try:
            title = plt.gca().get_title()
        except Exception:
            title = ""
        label = _sanitize_fig_name(title if title else "figure")
    else:
        label = _sanitize_fig_name(name)

    out = FIG_DIR / f"{_fig_counter:03d}_{label}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def savefig(name=None, force=True):
    return save_current_fig(name=name)

def show_save_close(name=None):
    save_current_fig(name)
    plt.show()
    plt.close()

print("Figure directory:", FIG_DIR)
print("Explicit save mode: plt.show is not patched.")


In [ ]:
from pathlib import Path

%matplotlib inline
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.spatial.distance import cdist
from matplotlib.colors import ListedColormap, to_rgba

os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    import nibabel as nib
    from nilearn.input_data import NiftiMasker
    from nilearn import plotting as niplot
    NILEARN_AVAILABLE = True
except Exception:
    NILEARN_AVAILABLE = False
    print("nilearn / nibabel not available; brain plotting disabled.")

print("Ready.")

In [ ]:
# ============================================================
# NOTEBOOK IDENTITY / SAVED DECODING SETTINGS
# ============================================================

ANALYSIS_TYPE = "spatial"
FIT_SCOPE = "within"

USE_SAVED_DECODING = True
DECODING_OUTPUT_DIR = "msaa_condrank_decoding_outputs_spatial_within"
PER_ARCH_DECODING_CSV = os.path.join(DECODING_OUTPUT_DIR, "per_archetype_mean_accuracy.csv")

USE_CACHE = True
OVERWRITE_CACHE = False

In [ ]:

FIT_LOAD_DIR = "msaa_flexible_outputs_npz"
DECODE_LOAD_DIR = "msaa_condrank_decoding_outputs_spatial_within"

CONDITIONS = ["intact", "word", "rest"]
K_VALUES = [88]

TOP_N_REPORT = 5

COND_COLORS = {"intact": "purple", "word": "green", "rest": "black"}

SCHAEFER_TXT = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order.txt"
SCHAEFER_NII = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz"
POSTERIOR_MAT = "data/pieman/raw/pieman_posterior_K700.mat"

OUTPUT_DIR = "005_postprocess_within_spatial_condition_specific_outputs"

In [ ]:
# ============================================================
# LOAD SAVED PER-ARCHETYPE DECODING
# ============================================================

def load_per_archetype_decoding(path=PER_ARCH_DECODING_CSV):
    if not os.path.exists(path):
        print("Saved per-archetype decoding CSV not found:", path)
        return pd.DataFrame()

    df = pd.read_csv(path)
    rename = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "sem" not in df.columns:
        rename["sem_accuracy"] = "sem"
    if "err" in df.columns and "sem" not in df.columns:
        rename["err"] = "sem"
    df = df.rename(columns=rename)

    if "analysis_type" in df.columns:
        df = df[df["analysis_type"].astype(str) == ANALYSIS_TYPE].copy()
    if "fit_scope" in df.columns:
        df = df[df["fit_scope"].astype(str) == FIT_SCOPE].copy()

    if "K" in df.columns:
        df["K"] = df["K"].astype(int)
    if "archetype" in df.columns:
        df["archetype"] = df["archetype"].astype(int)
    if "condition" in df.columns:
        df["condition"] = df["condition"].astype(str)

    print("Loaded saved per-archetype decoding:", path)
    print("Rows:", len(df))
    if len(df):
        display(df.head())
    return df

per_arch_decoding_df = load_per_archetype_decoding()

In [ ]:

def to_float_array(x):
    return np.array(x, dtype=float)

def load_msaa_npz(path):
    data = np.load(path, allow_pickle=True)
    results_subj = data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()
    return {
        "K": int(data["K"]),
        "results_subj": results_subj,
        "condition_labels_str": data["condition_labels_str"].tolist(),
        "condition_codes": data["condition_codes"] if "condition_codes" in data else None,
        "condition_names": data["condition_names"].tolist() if "condition_names" in data else None,
    }

fits = {}
for cond_name in CONDITIONS:
    fits[cond_name] = {}
    for K in K_VALUES:
        path = os.path.join(FIT_LOAD_DIR, f"spatialAA_within_{cond_name}_K{K}.npz")
        fits[cond_name][K] = load_msaa_npz(path)
        print("Loaded fit:", path)

rankings = np.load(os.path.join(DECODE_LOAD_DIR, "rankings.npy"), allow_pickle=True).item()
per_arch_df = pd.read_csv(os.path.join(DECODE_LOAD_DIR, "per_archetype_mean_accuracy.csv"))

example_sub = fits[CONDITIONS[0]][K_VALUES[0]]["results_subj"][0]
print("sXC shape:", np.asarray(example_sub["sXC"]).shape)
print("S shape:", np.asarray(example_sub["S"]).shape)

In [ ]:

posterior = loadmat(POSTERIOR_MAT)
centers = np.asarray(posterior['posterior']['centers'][0][0][0][0][0], dtype=float)
widths = np.asarray(list(posterior['posterior']['widths'][0][0][0][0][0][:, 0].T), dtype=float).ravel()

lookup_table = {
    'Vis': 'Visual',
    'SomMot': 'Somatomotor',
    'DorsAttn': 'Dorsal attention',
    'SalVentAttn': 'Ventral attention',
    'Limbic': 'Limbic',
    'Cont': 'Frontoparietal',
    'Default': 'Default mode'
}
network_colors = {
    'Visual': '#D7DF23',
    'Somatomotor': '#39B54A',
    'Dorsal attention': '#00A79D',
    'Ventral attention': '#27AAE1',
    'Limbic': '#1C75BC',
    'Frontoparietal': '#92278F',
    'Default mode': '#EE2A7B'
}
colors = ['#888888'] + [v for _, v in network_colors.items()]
network_cmap = ListedColormap(colors, N=len(colors) * 2, name='networks')
network_codes = {k: i + 1 for i, k in enumerate(lookup_table.values())}
print(network_codes)

In [ ]:

def nii2cmu(nifti_file, mask_file=None):
    def fullfact(dims):
        vals = np.asmatrix(range(1, dims[0] + 1)).T
        if len(dims) == 1:
            return vals
        aftervals = np.asmatrix(fullfact(dims[1:]))
        inds = np.asmatrix(np.zeros((np.prod(dims), len(dims))))
        row = 0
        for i in range(aftervals.shape[0]):
            inds[row:(row + len(vals)), 0] = vals
            inds[row:(row + len(vals)), 1:] = np.tile(aftervals[i, :], (len(vals), 1))
            row += len(vals)
        return inds

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        img = nib.load(nifti_file) if type(nifti_file) == str else nifti_file
        mask = NiftiMasker(mask_strategy='background')
        mask.fit(nifti_file if mask_file is None else mask_file)

    S = img.get_sform()
    Y = np.float32(mask.transform(nifti_file)).copy()
    vmask = np.nonzero(np.array(np.reshape(mask.mask_img_.dataobj, (1, np.prod(mask.mask_img_.shape)), order='C')))[1]
    vox_coords = fullfact(img.shape[0:3])[vmask, ::-1] - 1
    R = np.array(np.dot(vox_coords, S[0:3, 0:3])) + S[:3, 3]
    return {'Y': Y, 'R': R}

def rbf(R, center, width):
    return np.exp(-np.sum((R - center) ** 2, axis=1) / width)

def node_labels(centers, widths, networks_cmu):
    labels = []
    for c, w in zip(centers, widths):
        r = rbf(networks_cmu['R'], c, w)
        label_weights = [sum(r[networks_cmu['Y'].ravel() == i]) for i in range(1, len(network_codes) + 1)]
        labels.append(np.argmax(label_weights) + 1)
    return pd.DataFrame({
        'code': labels,
        'Network': [list(lookup_table.values())[i - 1] for i in labels]
    })

key = pd.read_csv(
    SCHAEFER_TXT,
    sep='\t',
    header=None,
    names=['id', 'name', 'x', 'y', 'z', 't']
).drop('t', axis=1)
key['study'] = key['name'].apply(lambda x: x.split('_')[0])
key['hemisphere'] = key['name'].apply(lambda x: x.split('_')[1][0])
key['network'] = key['name'].apply(lambda x: x.split('_')[2])
key.drop('name', axis=1, inplace=True)
key['network'] = key['network'].apply(lambda x: lookup_table[x])
key['code'] = key['network'].apply(lambda x: network_codes[x])
key.set_index('id', inplace=True)
key.loc[0, 'code'] = 0

networks_cmu = nii2cmu(SCHAEFER_NII)
networks_cmu['Y'] = np.atleast_2d(np.array([key.loc[i, 'code'] for i in networks_cmu['Y']]).astype(float))
node_codes_template = node_labels(centers, widths, networks_cmu)
display(node_codes_template.head())

In [ ]:

def get_top_archetypes_for_condition(cond_name, K, top_n=5):
    if (cond_name, K) in rankings:
        return list(rankings[(cond_name, K)][:top_n])

    sub = per_arch_df[(per_arch_df["condition"] == cond_name) & (per_arch_df["K"] == K)]
    if len(sub):
        return sub.sort_values("mean_accuracy", ascending=False)["archetype"].astype(int).tolist()[:top_n]

    raise ValueError(f"Could not find top archetypes for condition={cond_name}, K={K}")

def decoder(corrs):
    out = pd.DataFrame({'rank': [0.0], 'accuracy': [0.0], 'error': [0.0]})
    T = corrs.shape[0]
    for t in range(T):
        decoded_ind = int(np.argmax(corrs[t, :]))
        out.loc[0, 'error'] += np.mean(np.abs(decoded_ind - t)) / T
        out.loc[0, 'accuracy'] += (decoded_ind == t)
        out.loc[0, 'rank'] += np.mean((corrs[t, :] <= corrs[t, t]).astype(int))
    out['error'] /= T
    out['accuracy'] /= T
    out['rank'] /= T
    return out

def get_xval_assignments(ndata, nfolds, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    group_assignments = np.zeros(ndata, dtype=int)
    groupsize = int(np.ceil(ndata / nfolds))
    for i in range(1, nfolds):
        inds = np.arange(i * groupsize, min((i + 1) * groupsize, ndata))
        group_assignments[inds] = i
    rng.shuffle(group_assignments)
    return group_assignments

def build_single_archetype_recon_stack(subjects, k):
    Xhats = []
    for sub in subjects:
        sXC = np.asarray(sub["sXC"], dtype=float)
        S = np.asarray(sub["S"], dtype=float)
        Xhat = sXC[:, [k]] @ S[[k], :]
        Xhat = (Xhat - Xhat.mean(axis=0, keepdims=True)) / (Xhat.std(axis=0, keepdims=True) + 1e-8)
        Xhats.append(Xhat)
    return np.stack(Xhats, axis=0)

def recompute_single_archetype_decoding(subjects, k, nfolds=2, nreps=20, seed=42):
    recon_stack = build_single_archetype_recon_stack(subjects, k)
    N, T, V = recon_stack.shape
    rng = np.random.default_rng(seed)
    rows = []
    for rep in range(nreps):
        fold_ids = get_xval_assignments(N, nfolds, rng=rng)
        for i in range(nfolds):
            in_mask = (fold_ids == i)
            out_mask = ~in_mask
            in_mean = recon_stack[in_mask].mean(axis=0)
            out_mean = recon_stack[out_mask].mean(axis=0)
            corrs = 1.0 - cdist(in_mean, out_mean, metric='correlation')
            res = decoder(corrs)
            res["rep"] = rep
            res["fold"] = i
            rows.append(res)
    dec_df = pd.concat(rows, ignore_index=True)
    return {
        "mean": float(dec_df["accuracy"].mean()),
        "sem": float(dec_df["accuracy"].std() / np.sqrt(max(len(dec_df), 1))),
        "runs": dec_df
    }

In [ ]:

def plot_decoding_bar(cond_name, K, k, dec_mean, dec_sem):
    plt.figure(figsize=(4.5, 4))
    plt.bar([cond_name], [dec_mean], color=[COND_COLORS[cond_name]])
    plt.errorbar([0], [dec_mean], yerr=[dec_sem], fmt='none', color='black', capsize=4)
    plt.ylabel("Decoding accuracy")
    plt.title(f"{cond_name} | K={K} | archetype {k}")
    plt.tight_layout()
    save_current_fig()
    plt.show()
    plt.close()

def plot_temporal_motif(results_subj, cond_name, K, k):
    Xk = np.stack([to_float_array(sub["sXC"])[:, k] for sub in results_subj], axis=0)
    mean = Xk.mean(axis=0)
    sem = Xk.std(axis=0) / np.sqrt(max(Xk.shape[0], 1))
    x = np.arange(len(mean))
    plt.figure(figsize=(9, 4))
    plt.plot(x, mean, color=COND_COLORS[cond_name], label=cond_name)
    plt.fill_between(x, mean - sem, mean + sem, color=COND_COLORS[cond_name], alpha=0.2)
    plt.xlabel("Time")
    plt.ylabel("Archetype value")
    plt.title(f"{cond_name} | K={K} | archetype {k} | temporal motif")
    plt.legend()
    plt.tight_layout()
    save_current_fig()
    plt.show()
    plt.close()

def coefficient_to_alpha(weights, keep_mask, alpha_min=0.15, alpha_max=1.0):
    w = np.asarray(weights, dtype=float).ravel()
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    wk = w[keep_mask]
    if len(wk) == 0:
        return np.array([])
    wmin, wmax = wk.min(), wk.max()
    if np.isclose(wmax, wmin):
        return np.full(len(wk), alpha_max)
    scaled = (wk - wmin) / (wmax - wmin)
    return alpha_min + (alpha_max - alpha_min) * scaled

def plot_network_colored_spatial_coeff_map(results_subj, cond_name, K, k, title="", display_mode="lyrz", node_size=10, thr_frac=0.30, use_opacity=False, alpha_min=0.15, alpha_max=1.0):
    if not NILEARN_AVAILABLE:
        print("nilearn not available; skipping.")
        return None, None, None

    coeffs = np.stack([np.asarray(sub["S"], dtype=float)[k, :] for sub in results_subj], axis=0)
    weights = np.nan_to_num(coeffs.mean(axis=0), nan=0.0, posinf=0.0, neginf=0.0)
    wmax = np.max(weights)
    if wmax <= 0:
        print(f"Skipping archetype {k}: non-positive weights")
        return None, None, None

    keep = weights >= (thr_frac * wmax)
    centers_sel = centers[keep]
    node_codes_local = node_labels(centers, widths, networks_cmu)
    codes_sel = node_codes_local.loc[keep, 'code'].to_numpy()

    if use_opacity:
        alphas = coefficient_to_alpha(weights, keep, alpha_min=alpha_min, alpha_max=alpha_max)
        node_colors = []
        for code_idx, alpha in zip(codes_sel, alphas):
            base = to_rgba(colors[code_idx])
            node_colors.append((base[0], base[1], base[2], float(alpha)))
    else:
        node_colors = [colors[i] for i in codes_sel]

    disp = niplot.plot_connectome(
        np.eye(centers_sel.shape[0]),
        centers_sel,
        node_size=node_size,
        node_color=node_colors,
        display_mode=display_mode,
        title=title if title else f"{cond_name} | K={K} | archetype {k}"
    )
    save_current_fig()
    plt.show()
    plt.close()
    plt.clf()
    return disp, node_codes_local, keep

def plot_network_pie_for_selected_nodes(node_codes_local, keep_mask, title="Network composition"):
    selected = node_codes_local.loc[keep_mask].copy()
    counts = selected["Network"].value_counts().reindex(list(network_colors.keys()), fill_value=0)
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    pie_colors = [network_colors[name] for name in counts.index]
    ax.pie(
        counts.values,
        labels=counts.index,
        colors=pie_colors,
        autopct=lambda p: f"{p:.1f}%" if p > 0 else "",
        startangle=90,
        counterclock=False
    )
    ax.set_title(title)
    plt.tight_layout()
    save_current_fig()
    plt.show()
    plt.close()
    plt.close(fig)
    return counts

In [ ]:

summary_rows = []

for cond_name in CONDITIONS:
    for K in K_VALUES:
        print("\n" + "="*90)
        print(f"Condition={cond_name} | K={K}")
        print("="*90)

        results_subj = fits[cond_name][K]["results_subj"]
        top_arches = get_top_archetypes_for_condition(cond_name, K, top_n=TOP_N_REPORT)
        print("Top archetypes:", top_arches)

        for rank, k in enumerate(top_arches, start=1):
            dec_info = recompute_single_archetype_decoding(results_subj, k, nfolds=2, nreps=20, seed=42)
            summary_rows.append({
                "condition": cond_name,
                "K": K,
                "rank": rank,
                "archetype": int(k),
                "mean_accuracy": dec_info["mean"],
                "sem_accuracy": dec_info["sem"],
            })

            plot_decoding_bar(cond_name, K, k, dec_mean=dec_info["mean"], dec_sem=dec_info["sem"])
            plot_temporal_motif(results_subj, cond_name, K, k)

            disp1, node_codes_local, keep_mask = plot_network_colored_spatial_coeff_map(
                results_subj, cond_name, K, k,
                title=f"{cond_name} | K={K} | archetype {k} | network-colored coefficient plot",
                display_mode="lyrz",
                node_size=10,
                thr_frac=0.30,
                use_opacity=False
            )
            if disp1 is not None:
                try:
                    disp1.close()
                except Exception:
                    pass

            disp2, node_codes_local2, keep_mask2 = plot_network_colored_spatial_coeff_map(
                results_subj, cond_name, K, k,
                title=f"{cond_name} | K={K} | archetype {k} | opacity-weighted coefficient plot",
                display_mode="lyrz",
                node_size=10,
                thr_frac=0.30,
                use_opacity=True,
                alpha_min=0.15,
                alpha_max=1.0
            )
            if disp2 is not None:
                try:
                    disp2.close()
                except Exception:
                    pass

            if node_codes_local is not None and keep_mask is not None:
                plot_network_pie_for_selected_nodes(
                    node_codes_local,
                    keep_mask,
                    title=f"{cond_name} | K={K} | archetype {k} | selected-node network composition"
                )

summary_df = pd.DataFrame(summary_rows)
display(summary_df.head(20))

In [ ]:

summary_df.to_csv(
    os.path.join(OUTPUT_DIR, "spatial_within_top_archetypes_summary_clean_refactored.csv"),
    index=False
)
print("Saved summaries to:", OUTPUT_DIR)